In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df=pd.read_csv('../data/raw/fake_job_postings.csv')

In [3]:
df['has_company_logo'] = df['has_company_logo'].fillna(0)
df['has_questions'] = df['has_questions'].fillna(0)
df['telecommuting'] = df['telecommuting'].fillna(0)
df['employment_type']=df['employment_type'].fillna('Unknown')
df['employment_type'] = df['employment_type'].str.lower()

In [4]:
df['employment_type'].unique()

array(['other', 'full-time', 'unknown', 'part-time', 'contract',
       'temporary'], dtype=object)

In [5]:
employment_dummies = pd.get_dummies(
    df['employment_type'],
)

df = pd.concat([df, employment_dummies], axis=1)


In [6]:
df.head()

,job_id,title,location,department,salary_range,company_profile,description,requirements,benefits,telecommuting,...,required_education,industry,function,fraudulent,contract,full-time,other,part-time,temporary,unknown
0,1,Marketing Intern,"US, NY, New York",Marketing,NaN,"We're Food52, and we've created a groundbreaki...","Food52, a fast-growing, James Beard Award-winn...",Experience with content management systems a m...,NaN,0,...,NaN,NaN,Marketing,0,False,False,True,False,False,False
1,2,Customer Service - Cloud Video Production,"NZ, , Auckland",Success,NaN,"90 Seconds, the worlds Cloud Video Production ...",Organised - Focused - Vibrant - Awesome!Do you...,What we expect from you:Your key responsibilit...,What you will get from usThrough being part of...,0,...,NaN,Marketing and Advertising,Customer Service,0,False,True,False,False,False,False
2,3,Commissioning Machinery Assistant (CMA),"US, IA, Wever",NaN,NaN,Valor Services provides Workforce Solutions th...,"Our client, located in Houston, is actively se...",Implement pre-commissioning and commissioning ...,NaN,0,...,NaN,NaN,NaN,0,False,False,False,False,False,True
3,4,Account Executive - Washington DC,"US, DC, Washington",Sales,NaN,Our passion for improving quality of life thro...,THE COMPANY: ESRI – Environmental Systems Rese...,"EDUCATION: Bachelor’s or Master’s in GIS, busi...",Our culture is anything but corporate—we have ...,0,...,Bachelor's Degree,Computer Software,Sales,0,False,True,False,False,False,False
4,5,Bill Review Manager,"US, FL, Fort Worth",NaN,NaN,SpotSource Solutions LLC is a Global Human Cap...,JOB TITLE: Itemization Review ManagerLOCATION:...,QUALIFICATIONS:RN license in the State of Texa...,Full Benefits Offered,0,...,Bachelor's Degree,Hospital & Health Care,Health Care Provider,0,False,True,False,False,False,False


In [7]:
df['company_profile_length'] = df['company_profile'].fillna('').apply(len)

In [9]:
import pandas as pd
import spacy

# Load spaCy model (disable components you don't need for speed)
nlp = spacy.load("en_core_web_sm", disable=["tagger", "parser"])

# Ensure columns exist
df['org_count'] = 0
df['gpe_count'] = 0

# Convert company_profile column to list (replace NaN with empty string)
texts = df['company_profile'].fillna("").tolist()

# Prepare lists to store counts
org_counts = []
gpe_counts = []

# Process texts in batches using nlp.pipe (MUCH faster than looping)
for doc in nlp.pipe(texts, batch_size=50):
    org_count = sum(1 for ent in doc.ents if ent.label_ == "ORG")
    gpe_count = sum(1 for ent in doc.ents if ent.label_ == "GPE")
    org_counts.append(org_count)
    gpe_counts.append(gpe_count)

# Assign results back to DataFrame
df['org_count'] = org_counts
df['gpe_count'] = gpe_counts


C:\Users\Akshay\AppData\Roaming\Python\Python313\site-packages\spacy\pipeline\lemmatizer.py:188: UserWarning: [W108] The rule-based lemmatizer did not find POS annotation for one or more tokens. Check that your pipeline includes components that assign token.pos, typically 'tagger'+'attribute_ruler' or 'morphologizer'.
  warnings.warn(Warnings.W108)


In [10]:
df['company_profile_length_log'] = np.log1p(df['company_profile_length'])
df['org_count_log'] = np.log1p(df['org_count'])
df['gpe_count_log'] = np.log1p(df['gpe_count'])

In [11]:
df['company_profile_missing'] = df['company_profile'].isna().astype(int)
df['required_experience_missing'] = df['required_experience'].isna().astype(int)


In [12]:
df['info_completeness_score'] = (
    (1 - df['company_profile_missing']) +
    (1 - df['required_experience_missing']) +
    df['company_profile_length_log'] +
    df['org_count_log'] +
    df['gpe_count_log']
)


In [13]:
df.groupby('fraudulent')['info_completeness_score'].mean()


fraudulent
0    8.075436
1    3.322488
Name: info_completeness_score, dtype: float64

In [14]:
urgency_keywords = [
    'immediate', 'urgent', 'asap',
    'quickly', 'fast', 'fast hiring', 'limited'
]
for i in range(len(df)):
    text = df.loc[i, 'description']
    
    if pd.isna(text):
        df.loc[i, 'has_urgency_language'] = 0 
        continue
    
    text = text.lower()
    df['has_urgency_language'] = 0
    for word in urgency_keywords:
        if word in text:
            df.loc[i, 'has_urgency_language'] = 1
            break


In [15]:
df['lexical_diversity']=0.0
for i in range(len(df)):
    text=df.loc[i,'description']
    if pd.isna(text):
        continue
    words=text.lower().split()
    if(len(words)==0):
        continue
    else:
        unique_words=set(words)
        df.loc[i,'lexical_diversity']=len(unique_words)/len(words)

In [16]:
df.columns

Index(['job_id', 'title', 'location', 'department', 'salary_range',
       'company_profile', 'description', 'requirements', 'benefits',
       'telecommuting', 'has_company_logo', 'has_questions', 'employment_type',
       'required_experience', 'required_education', 'industry', 'function',
       'fraudulent', 'contract', 'full-time', 'other', 'part-time',
       'temporary', 'unknown', 'company_profile_length', 'org_count',
       'gpe_count', 'company_profile_length_log', 'org_count_log',
       'gpe_count_log', 'company_profile_missing',
       'required_experience_missing', 'info_completeness_score',
       'has_urgency_language', 'lexical_diversity'],
      dtype='object')

In [17]:
feature_columns = ['company_profile_length_log','org_count_log','gpe_count_log','company_profile_missing','required_experience_missing','info_completeness_score',
                   'has_urgency_language','has_company_logo','contract', 'full-time', 'other', 'part-time',
                   'temporary', 'unknown','has_questions','telecommuting']

In [18]:
df_final = df[feature_columns + ['fraudulent']]


In [19]:
df_final.shape

(17880, 17)

In [20]:
df_final.isna().sum()

company_profile_length_log     0
org_count_log                  0
gpe_count_log                  0
company_profile_missing        0
required_experience_missing    0
info_completeness_score        0
has_urgency_language           0
has_company_logo               0
contract                       0
full-time                      0
other                          0
part-time                      0
temporary                      0
unknown                        0
has_questions                  0
telecommuting                  0
fraudulent                     0
dtype: int64

In [21]:
bool_cols = df_final.select_dtypes(include='bool').columns
df_final[bool_cols] = df_final[bool_cols].astype(int)


C:\Users\Akshay\AppData\Local\Temp\ipykernel_12200\632429486.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_final[bool_cols] = df_final[bool_cols].astype(int)


In [22]:
df_final.head()

,company_profile_length_log,org_count_log,gpe_count_log,company_profile_missing,required_experience_missing,info_completeness_score,has_urgency_language,has_company_logo,contract,full-time,other,part-time,temporary,unknown,has_questions,telecommuting,fraudulent
0,6.786717,2.302585,0.693147,0,0,11.782449,0,1,0,0,1,0,0,0,0,0,0
1,7.160069,1.791759,1.609438,0,0,12.561267,0,1,0,1,0,0,0,0,0,0,0
2,6.779922,2.197225,1.098612,0,1,11.075759,0,1,0,0,0,0,0,1,0,0,0
3,6.421622,1.098612,0.000000,0,0,9.520235,0,1,0,1,0,0,0,0,0,0,0
4,7.395722,2.944439,1.098612,0,0,13.438773,0,1,0,1,0,0,0,0,1,0,0


In [39]:
df_final.to_csv(
    "../data/processed/job_fraud_features.csv",
    index=False
)


In [40]:
df_final.groupby('fraudulent')['info_completeness_score'].median()

fraudulent
0    9.063048
1    1.000000
Name: info_completeness_score, dtype: float64

In [41]:
df_final.groupby('fraudulent')[['company_profile_missing', 'org_count_log']].mean()


,company_profile_missing,org_count_log
fraudulent,,
0,0.159927,0.815305
1,0.677829,0.347735


In [23]:
df['title'].unique()

array(['Marketing Intern', 'Customer Service - Cloud Video Production',
       'Commissioning Machinery Assistant (CMA)', ...,
       'Senior Financial Analyst (Retail) ',
       'Account Director - Distribution ',
       'Project Cost Control Staff Engineer - Cost Control Exp - TX'],
      shape=(11231,), dtype=object)

In [27]:
junior_keywords=['junior','jr','entry level','entry-level','intern','internship','trainee','associate','assistant']
senior_keywords=['senior','sr','lead','manager','director','principal','head','chief','officer','executive']

df['title_seniority']='unknown'

for i in range(len(df)):
    title=df.loc[i,'title']
    if pd.isna(title):
        continue
    title=title.lower()

    for word in junior_keywords:
        if word in title:
            df.loc[i,'title_seniority']='junior'
            break
    if df.loc[i,'title_seniority']=='unknown':
        for word in senior_keywords:
            if word in title:
                df.loc[i,'title_seniority']='senior'
                break


In [28]:
df['title_seniority'].value_counts()


title_seniority
unknown    10784
senior      4715
junior      2381
Name: count, dtype: int64

In [29]:
df[['title', 'title_seniority']].head(10)


,title,title_seniority
0,Marketing Intern,junior
1,Customer Service - Cloud Video Production,unknown
2,Commissioning Machinery Assistant (CMA),junior
3,Account Executive - Washington DC,senior
4,Bill Review Manager,senior
5,Accounting Clerk,unknown
6,Head of Content (m/f),senior
7,Lead Guest Service Specialist,senior
8,HP BSM SME,unknown
9,Customer Service Associate - Part Time,junior


In [30]:
df['required_experience'].unique()

array(['Internship', 'Not Applicable', nan, 'Mid-Senior level',
       'Associate', 'Entry level', 'Executive', 'Director'], dtype=object)

In [33]:
mapping={'Internship':'junior','Entry level':'junior','Associate':'junior','Mid-Senior level':'mid','Director':'senior','Executive':'senior','Not Application':'unknown','nan':'unknown'}
df['experience_seniority']=df['required_experience'].map(mapping)

In [34]:
df['experience_seniority'].value_counts()


experience_seniority
junior    5375
mid       3809
senior     530
Name: count, dtype: int64

In [38]:
df['experience_role_mismatch']=((df['experience_seniority']=='senior') & (df['title_seniority']=='junior')).astype(int)

In [39]:
df['experience_role_mismatch'].value_counts()

experience_role_mismatch
0    17868
1       12
Name: count, dtype: int64

In [4]:
generic_words=['dynamic','fast-paced','innovative','growth','opportunity','leading','global','excellence','passionate','motivated','success',
'culture','vision','world-class']
df['generic_word_ratio'] = 0.0

for i in range(len(df)):
    text = df.loc[i, 'description']
    
    if pd.isna(text):
        continue
        
    text = text.lower().split()
    
    generic_count = 0
    for word in generic_words:
        generic_count += text.count(word)
    
    if len(text) > 0:
        df.loc[i, 'generic_word_ratio'] = generic_count / len(text)


In [6]:
df.groupby('fraudulent')['generic_word_ratio'].mean()

fraudulent
0    0.005371
1    0.004031
Name: generic_word_ratio, dtype: float64